# 第 1 周末练习 —— 自然语言转 SQL

## 练习目标（理念）

构建一个小工具：输入**关于数据的自然语言问题**，输出**可执行的 SQL**。  
同一套 schema + question，分别用 **OpenAI**（`gpt-4o-mini`）与 **Ollama**（`llama3.2`）生成，方便对比风格与正确性。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | system 定「只输出 SQL」；user 塞 Schema + Question |
| 流式输出 `stream=True` | GPT 路径逐块 `print(..., end="")` |
| OpenAI 云端模型 | 常量 `MODEL_GPT = 'gpt-4o-mini'` |
| Ollama 本地模型 | `ollama.chat(...)` + `MODEL_LLAMA = 'llama3.2'` |

## 怎么跑

1. 准备 `.env` 里的 `OPENAI_API_KEY`；本地需已 `ollama pull llama3.2`
2. 从上到下运行：导入与验钥 → 常量 → 改 `SCHEMA`/`question` → 跑 GPT 流式格 → 跑 Llama 格
3. 对比两段 SQL：是否只用了给定表字段、聚合/排序是否合理


In [ ]:
# ========== 导入 + 加载密钥 + 创建 OpenAI 客户端 ==========

# 导入标准库 os：读环境变量 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具（本练习主要用 print；保留原导入以免改逻辑）
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI：调用云端 Chat Completions
from openai import OpenAI
# 导入 ollama 官方 Python 包：后面用 ollama.chat 调本地模型
import ollama

# 加载 .env；override=True 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 API Key，做课程里常见的格式体检（print 文案保持英文原文）
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

# 默认 OpenAI 客户端（云端）；密钥来自环境变量
client = OpenAI()


In [ ]:
# ========== 常量：云端与本地模型名集中管理 ==========

# OpenAI 云端小模型：便宜、适合这种结构化短输出
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；须与本机已安装名一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 输入：表结构 SCHEMA + 自然语言问题 question ==========

# 自然语言问题 - 改 question 即可生成不同 SQL
# 可选：提供架构，以便模型生成准确的查询（字段名写清楚，减少胡编列名）

# SCHEMA：用 SQL 注释风格描述表与字段；发给模型时保持英文/标识符原样
SCHEMA = """
-- users: id (int), name (text), email (text), created_at (timestamp)
-- orders: id (int), user_id (int), total (decimal), created_at (timestamp)
-- products: id (int), name (text), price (decimal)
"""

# question：自然语言业务问题；保留英文，与课程示例一致、也避免改变模型输出习惯
question = "List the top 5 users by total order amount, with their total spent."


In [ ]:
# ========== OpenAI 路径：流式生成 SQL（只输出查询本身） ==========

# system_prompt：约束「只输出合法 SQL、不要解释/代码围栏」；保留英文以免改变行为
system_prompt = """You are a SQL expert. Given a database schema and a natural language question, output only a valid SQL query. No explanation, no markdown fences—just the SQL."""

# stream=True：返回可迭代的 chunk 流，而不是一次性完整字符串
stream = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": system_prompt},
        # user：把 SCHEMA 与 question 拼在一起，模型据此写 SQL
        {"role": "user", "content": f"Schema:\n{SCHEMA}\n\nQuestion: {question}"},
    ],
    stream=True,
)

# 提示当前是哪条后端的结果（展示用英文保持原样）
print("GPT-4o-mini SQL:\n")
# 逐块取出 delta.content；end="" 让 token 紧挨着打印，形成流式效果
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
# 流结束后补一个换行，避免下一个输出粘在同一行
print()


In [ ]:
# ========== Ollama 路径：非流式 chat，便于和 GPT 结果并排对比 ==========

# ollama.chat：本地推理；messages 结构与 OpenAI 类似（system + user）
response = ollama.chat(
    model=MODEL_LLAMA,
    messages=[
        # 复用上一格定义的 system_prompt，保证两边约束一致
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Schema:\n{SCHEMA}\n\nQuestion: {question}"},
    ],
)

# 打印 Llama 完整回复；路径是 response["message"]["content"]（dict 风格，不是 SDK 对象）
print("Llama 3.2 SQL:\n")
print(response["message"]["content"])


In [ ]:
# （空单元格）原笔记本此处无代码。
# 若要扩展：可把两边 SQL 再交给模型做「正确性点评」，或接到真实 SQLite 执行验证。
# 逻辑保持原样：不新增可执行调用，避免改变本练习的最小对比流程。
